# 04 - Prediction

Memuat model terbaru dari MLflow registry dan menulis gold predictions.

In [ ]:
%pip install mlflow lightgbm azureml-mlflow

In [ ]:
predict_days = "7"

In [ ]:
# Parameters
predict_days = int(predict_days)

# Configuration
import os
from datetime import datetime, timedelta, timezone

import pandas as pd
from pyspark.sql import functions as F

STORAGE_ACCOUNT = "sthidrogkiq7v7oc3lns"
GOLD = f"abfss://gold@{STORAGE_ACCOUNT}.dfs.core.windows.net"
MLFLOW_TRACKING_URI = os.getenv(
    "MLFLOW_TRACKING_URI",
    "https://eastasia.api.azureml.ms/mlflow/v1.0/subscriptions/838d94c7-d2fd-4c8c-8ceb-ad3ce9dfc707/resourceGroups/rg-hidrologi-v0/providers/Microsoft.MachineLearningServices/workspaces/mlw-hidrologi-v0",
)
STATIONS = ["11525670", "11150500", "11454000"]
HORIZONS = (1, 6, 24)
FEATURE_COLS = [
    "stage_now",
    "stage_lag_1", "stage_lag_2", "stage_lag_3", "stage_lag_6",
    "stage_lag_12", "stage_lag_24", "stage_lag_48", "stage_lag_72", "stage_lag_168",
    "stage_mean_3", "stage_mean_6", "stage_mean_12", "stage_mean_24", "stage_mean_72", "stage_mean_168",
    "stage_max_3", "stage_max_6", "stage_max_12", "stage_max_24", "stage_max_72", "stage_max_168",
    "stage_diff_1", "stage_diff_3", "stage_diff_6", "stage_diff_24",
    "discharge_lag_1", "discharge_lag_3", "discharge_lag_6", "discharge_lag_24", "discharge_lag_72",
    "precip_sum_1", "precip_sum_3", "precip_sum_6", "precip_sum_24", "precip_sum_72", "precip_sum_168",
    "precip_sum_24_lag_24", "precip_sum_72_lag_24",
    "soil_moist_0_7", "soil_moist_7_28", "temp_c", "et0_mm", "rh_pct",
    "hour_sin", "hour_cos", "doy_sin", "doy_cos",
]

import mlflow

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
features = spark.read.format("delta").load(f"{GOLD}/delta/features")
features = features.withColumn("time_utc", F.col("time_utc").cast("timestamp"))
latest = features.agg(F.max("time_utc")).first()[0]
since = latest - timedelta(days=predict_days)
frame = features.filter(F.col("time_utc") >= since).toPandas()
frame["time_utc"] = pd.to_datetime(frame["time_utc"], utc=True)
frame["station_id"] = frame["station_id"].astype(str)

records = []
for site in STATIONS:
    part = frame[frame["station_id"] == site].copy()
    if part.empty:
        continue
    for horizon in HORIZONS:
        name = f"hydrology-{site}-h{horizon}"
        try:
            model = mlflow.lightgbm.load_model(f"models:/{name}/latest")
        except Exception as exc:
            print(f"skip {name}: {exc}")
            continue
        columns = list(getattr(model, "feature_name_", FEATURE_COLS))
        usable = part.dropna(subset=[column for column in columns if column in part.columns] + ["stage_now"])
        if usable.empty:
            continue
        prediction = model.predict(usable[columns]) + usable["stage_now"].to_numpy()
        output = pd.DataFrame({
            "station_id": site,
            "issued_at": usable["time_utc"].to_numpy(),
            "target_time": usable["time_utc"].to_numpy() + timedelta(hours=horizon),
            "horizon": horizon,
            "y_pred": prediction,
            "y_obs": usable[f"y_h{horizon}"].to_numpy(),
            "model_version": name,
            "dataset_version": usable["dataset_version"].astype(str).to_numpy(),
            "created_at": datetime.now(timezone.utc),
        })
        output["issue_date"] = pd.to_datetime(output["issued_at"], utc=True).dt.strftime("%Y-%m-%d")
        records.append(output.dropna(subset=["y_pred"]))

if not records:
    print("no model available; prediction skipped")
else:
    combined = pd.concat(records, ignore_index=True)
    spark_frame = spark.createDataFrame(combined)
    issue_dates = pd.to_datetime(combined["issue_date"], utc=True)
    predicate = f"issue_date >= '{issue_dates.min().date().isoformat()}' AND issue_date <= '{issue_dates.max().date().isoformat()}'"
    writer = (
        spark_frame.write.format("delta").mode("overwrite")
        .partitionBy("station_id", "issue_date")
    )
    try:
        from delta.tables import DeltaTable
        if DeltaTable.isDeltaTable(spark, f"{GOLD}/delta/predictions"):
            writer = writer.option("replaceWhere", predicate)
    except Exception:
        pass
    writer.save(f"{GOLD}/delta/predictions")
    print(f"wrote {len(combined)} prediction rows")
